# Setup and reload

In [2]:
import sys, os
from pathlib import Path
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from config.settings import settings
from medrag.embeddings.qdrant_client import get_qdrant_client
from medrag.retrieval.hybrid_search import hybrid_search

client = get_qdrant_client(settings.qdrant_url or "http://localhost:6333")
client.get_collections()

from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded")

Project root: C:\Users\DELL\Desktop\medrag


c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Use pytorch device: cpu


Cross-encoder loaded


# Retrieve a wider candidate pool, then rerank with the cross-encoder

In [3]:
query_text = "how does the body regulate blood sugar"

# Step 1: retrieve a wider candidate pool via hybrid search
candidates = hybrid_search(client, query_text, limit=20, per_signal_limit=20)
print(f"Retrieved {len(candidates)} candidates via hybrid search")

# Step 2: rerank with the cross-encoder
pairs = [(query_text, c["payload"]["raw_text"]) for c in candidates]
cross_scores = cross_encoder.predict(pairs)

for c, score in zip(candidates, cross_scores):
    c["cross_score"] = float(score)

reranked = sorted(candidates, key=lambda c: c["cross_score"], reverse=True)

print("\n=== BEFORE reranking (hybrid RRF order) ===")
for c in candidates[:5]:
    print(f"  fused={c['fused_score']:.5f}  source={c['payload']['source']}  chunk_id={c['chunk_id']}")

print("\n=== AFTER reranking (cross-encoder order) ===")
for c in reranked[:5]:
    print(f"  cross_score={c['cross_score']:.4f}  source={c['payload']['source']}  chunk_id={c['chunk_id']}")
    print(f"    text: {c['payload']['raw_text'][:150]}")

Loading sparse model 'Qdrant/bm25'...


Retrieved 20 candidates via hybrid search


Batches: 100%|██████████| 1/1 [00:09<00:00,  9.32s/it]


=== BEFORE reranking (hybrid RRF order) ===
  fused=0.02914  source=who  chunk_id=diabetes_who_text_9
  fused=0.02582  source=openfda  chunk_id=Glipizide_openfda_12
  fused=0.01639  source=who  chunk_id=coronary artery disease+heart failure+hyperlipidemia+stroke_who_text_114
  fused=0.01639  source=who  chunk_id=asthma+copd_who_text_21
  fused=0.01613  source=openfda  chunk_id=Glipizide_openfda_11

=== AFTER reranking (cross-encoder order) ===
  cross_score=-1.6356  source=who  chunk_id=coronary artery disease+heart failure+hyperlipidemia+stroke_who_text_114
    text: In a meta-analysis of non-diabetic
subjects, those with the highest blood glucose levels had a relative risk for cardiovascular disease
events of 1.26
  cross_score=-4.0736  source=openfda  chunk_id=Glipizide_openfda_11
    text: Mechanism of Action The primary mode of action of glipizide in experimental animals appears to be the stimulation of insulin secretion from the beta c
  cross_score=-4.2915  source=openfda  chun

# Confirm reranking doesn't break an already-good query

In [4]:
query_text = "metformin dosage for diabetes"

candidates = hybrid_search(client, query_text, limit=20, per_signal_limit=20)
pairs = [(query_text, c["payload"]["raw_text"]) for c in candidates]
cross_scores = cross_encoder.predict(pairs)
for c, score in zip(candidates, cross_scores):
    c["cross_score"] = float(score)
reranked = sorted(candidates, key=lambda c: c["cross_score"], reverse=True)

print("=== BEFORE reranking ===")
for c in candidates[:5]:
    print(f"  fused={c['fused_score']:.5f}  chunk_id={c['chunk_id']}")

print("\n=== AFTER reranking ===")
for c in reranked[:5]:
    print(f"  cross_score={c['cross_score']:.4f}  chunk_id={c['chunk_id']}")

Batches: 100%|██████████| 1/1 [00:12<00:00, 12.46s/it]

=== BEFORE reranking ===
  fused=0.03126  chunk_id=Metformin Hydrochloride_openfda_0
  fused=0.03102  chunk_id=Metformin Hydrochloride_openfda_4
  fused=0.03068  chunk_id=Synjardy_openfda_26
  fused=0.02838  chunk_id=Metformin Hydrochloride_openfda_18
  fused=0.02740  chunk_id=Metformin Hydrochloride_openfda_13

=== AFTER reranking ===
  cross_score=7.5538  chunk_id=Synjardy_openfda_26
  cross_score=5.4592  chunk_id=Metformin Hydrochloride_openfda_1
  cross_score=4.6757  chunk_id=Metformin Hydrochloride_openfda_2
  cross_score=3.8993  chunk_id=Jardiance_openfda_23
  cross_score=3.8951  chunk_id=Synjardy_openfda_34


# Wrap into a rerank() function and a combined search_with_reranking()

In [5]:
def rerank(query_text: str, candidates: list, top_n: int = 5) -> list:
    """Score each candidate's raw_text against the query with the
    cross-encoder and return the top_n by that score, descending."""
    pairs = [(query_text, c["payload"]["raw_text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)
    for c, score in zip(candidates, scores):
        c["cross_score"] = float(score)
    return sorted(candidates, key=lambda c: c["cross_score"], reverse=True)[:top_n]


def search_with_reranking(query_text: str, candidate_pool_size: int = 20, top_n: int = 5) -> list:
    candidates = hybrid_search(client, query_text, limit=candidate_pool_size, per_signal_limit=candidate_pool_size)
    return rerank(query_text, candidates, top_n=top_n)


# quick end-to-end sanity check on a third, fresh query
results = search_with_reranking("side effects of ACE inhibitors")
for r in results:
    print(f"cross_score={r['cross_score']:.4f}  source={r['payload']['source']}  chunk_id={r['chunk_id']}")
    print(f"  text: {r['payload']['raw_text'][:150]}")

Batches: 100%|██████████| 1/1 [00:04<00:00,  4.53s/it]

cross_score=6.1535  source=who  chunk_id=coronary artery disease+heart failure+hyperlipidemia+stroke_who_table_263
  text: Class of drug | Major adverse effects
ACE inhibitors | dry cough, renal dysfunction in patients with impaired renal function
ARBs | increase in hepati
cross_score=5.9202  source=openfda  chunk_id=Everolimus_openfda_68
  text: 7.2 Effects of Combination Use of Angiotensin Converting Enzyme (ACE) Inhibitors Patients taking concomitant ACE inhibitors with everolimus tablets/ev
cross_score=5.2322  source=openfda  chunk_id=Ramipril_openfda_6
  text: 5. WARNINGS AND PRECAUTIONS ACE inhibitor use has been associated with the following: • Angioedema, with increased risk in patients with a prior histo
cross_score=5.1469  source=openfda  chunk_id=Trandolapril_openfda_12
  text: Non-Steroidal Anti-Inflammatory Agents including Selective Cyclooxygenase-2 Inhibitors (COX-2 Inhibitors) In patients who are elderly, volume-depleted
cross_score=4.1885  source=openfda  chunk_id=Rami